In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [4]:
import pandas as pd
df = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_orders_dataset.csv')

print(f"Số dòng: {df.shape[0]}, số cột: {df.shape[1]}")

print(f"Tên các cột:")
print(df.columns.to_list())

print("Các giá trị thiếu của từng cột: ")
print(df.isna().sum())

print(f"Kiểu dữ liệu của từng cột: ")
print(df.dtypes)


Số dòng: 99441, số cột: 8
Tên các cột:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Các giá trị thiếu của từng cột: 
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
Kiểu dữ liệu của từng cột: 
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


99,441 đơn hàng;

3 cột bị missing: order_approved_at (160), order_delivered_carrier_date (1783), order_delivered_customer_date (2965) — có đơn hàng bị hủy hoặc chưa giao;

tất cả cột ngày đang là object thay vì datetime — cần fix trước khi phân tích



In [5]:
#fix kiểu dữ liệu ngày
date_col = ['order_purchase_timestamp',
            'order_approved_at','order_delivered_carrier_date',
'order_delivered_carrier_date',
'order_delivered_customer_date',
'order_estimated_delivery_date']
for col in date_col:
  df[col] = pd.to_datetime(df[col])

print("kết quả: ")
print(df.dtypes)

kết quả: 
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [6]:
# các trạng thái đơn hàng
print(df['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


96,478/99,441 đơn hàng đã giao thành công

Khi phân tích doanh thu và hành vi khách hàng, chỉ giữ lại đơn delivered — các đơn canceled, unavailable sẽ làm lệch kết quả.

In [7]:
df_d = df[df['order_status']=='delivered'].copy()

print(f"Trước khi lọc: {df.shape[0]} dòng")
print(f"sau khi lọc: {df_d.shape[0]} dòng")

Trước khi lọc: 99441 dòng
sau khi lọc: 96478 dòng


In [8]:
# thêm các cột thời gian
df_d['year'] = df_d['order_purchase_timestamp'].dt.year
df_d['month'] = df_d['order_purchase_timestamp'].dt.month
df_d['day_of_week'] = df_d['order_purchase_timestamp'].dt.day_name()
print(df_d[['order_purchase_timestamp','year','month','day_of_week']].head(5))

  order_purchase_timestamp  year  month day_of_week
0      2017-10-02 10:56:33  2017     10      Monday
1      2018-07-24 20:41:37  2018      7     Tuesday
2      2018-08-08 08:38:49  2018      8   Wednesday
3      2017-11-18 19:28:06  2017     11    Saturday
4      2018-02-13 21:18:39  2018      2     Tuesday


In [9]:
# tính thời gian giao hàng
df_d['d_day']=(
    df_d['order_delivered_customer_date'] -
    df_d['order_purchase_timestamp']
).dt.days

print(df_d['d_day'].describe())


count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: d_day, dtype: float64


trung bình mất 12 ngày để giao hàng (khá chậm)

nhanh nhất là giao trong ngày

chậm nhất là 209 ngày(gần 7 tháng, giá trị outlier bất thưởng)


In [10]:
monthly_orders = (
    df_d.groupby(['year','month'])
    .agg(total_orders = ('order_id','count'))
    .reset_index()
    .sort_values(['year','month'])
)

print(monthly_orders)
best_m = monthly_orders.loc[monthly_orders['total_orders'].idxmax()]
print(f"tháng có nhiều đơn nhất: {int(best_m['month'])}/{int(best_m['year'])}: {int(best_m['total_orders'])} đơn")


    year  month  total_orders
0   2016      9             1
1   2016     10           265
2   2016     12             1
3   2017      1           750
4   2017      2          1653
5   2017      3          2546
6   2017      4          2303
7   2017      5          3546
8   2017      6          3135
9   2017      7          3872
10  2017      8          4193
11  2017      9          4150
12  2017     10          4478
13  2017     11          7289
14  2017     12          5513
15  2018      1          7069
16  2018      2          6555
17  2018      3          7003
18  2018      4          6798
19  2018      5          6749
20  2018      6          6099
21  2018      7          6159
22  2018      8          6351
tháng có nhiều đơn nhất: 11/2017: 7289 đơn


 tháng 11 có lượng đơn lớn nhất có thể do black friday

 tăng trưởng mạnh theo năm từ 2016 đến 2018, với đỉnh cao vào tháng 11/2017 có thể liên quan đến Black Friday. Từ đầu 2018 tăng trưởng bắt đầu ổn định.





In [11]:
df_payments = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_payments_dataset.csv')
print(df_payments.head())
print(df_payments.shape)
print(df_payments['payment_type'].value_counts())


                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  
(103886, 5)
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


số lượng dòng nhiều hơn đơn hàng có thể do có nhiều payment(trả nhiều lần  hoặc voucher+thẻ)

credit card chiếm 74% tiền mặt(boleto) 19%

In [12]:
df_merged = df_d.merge(df_payments,on='order_id',how='left')
print(df_merged.columns.to_list())
# doanh thu theo tháng
monthly_revenue = (
    df_merged.groupby(['year','month'])
    .agg(revenue=('payment_value','sum'))
    .reset_index()
    .sort_values(['year','month'])
)
print(f'doanh thu theo thang: {monthly_revenue}')

# thang co doanh thu cao nhat
best_m_revenue = monthly_revenue.loc[monthly_revenue['revenue'].idxmax()]
print(f'thang co doanh thu cao nhat: {int(best_m_revenue['month'])}/{int(best_m_revenue['year'])}: {best_m_revenue['revenue']}')

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'year', 'month', 'day_of_week', 'd_day', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
doanh thu theo thang:     year  month     revenue
0   2016      9        0.00
1   2016     10    46566.71
2   2016     12       19.62
3   2017      1   127545.67
4   2017      2   271298.65
5   2017      3   414369.39
6   2017      4   390952.18
7   2017      5   567066.73
8   2017      6   490225.60
9   2017      7   566403.93
10  2017      8   646000.61
11  2017      9   701169.99
12  2017     10   751140.27
13  2017     11  1153528.05
14  2017     12   843199.17
15  2018      1  1078606.86
16  2018      2   966510.88
17  2018      3  1120678.00
18  2018      4  1132933.95
19  2018      5  1128836.69
20  2018      6  1012090.68
21  2018      7  1027903.86
22  2018      8   98541

tháng 11/2017 vừa có nhiều đơn nhất vừa có doanh thu cao nhất


In [13]:
# load bảng product
df_products = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_products_dataset.csv')
df_items = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_items_dataset.csv')
# merge items với products
df_items_products = df_items.merge(df_products[['product_id','product_category_name']],on='product_id',how='left')

# top 10 danh mục theo số lượng đơn
top_categories = (
    df_items_products.groupby('product_category_name')
    .agg(total_orders=('order_id','count'))
    .reset_index()
    .sort_values('total_orders',ascending=False)
    .head(10)
)
print(top_categories)



     product_category_name  total_orders
13         cama_mesa_banho         11115
11            beleza_saude          9670
32           esporte_lazer          8641
54        moveis_decoracao          8334
44  informatica_acessorios          7827
72   utilidades_domesticas          6964
66      relogios_presentes          5991
70               telefonia          4545
40      ferramentas_jardim          4347
8               automotivo          4235


cama_mesa_banho  là danh mục có nhiều lượng mua hàng nhất


In [14]:
# doanh thu theo danh mục
# merged  bảng orders để lọc delivered
df_item_product_order = df_items_products.merge(df_d[['order_id']],on='order_id',how='inner'
).merge(df_payments[['order_id','payment_value']],on='order_id',how='left')

# doanh thu theo danh muc
category_revenue =(
    df_item_product_order.groupby('product_category_name')
    .agg(revenue=('payment_value','sum'))
    .reset_index()
    .sort_values('revenue',ascending=False)
)
print(category_revenue)

            product_category_name     revenue
13                cama_mesa_banho  1692714.28
11                   beleza_saude  1620684.04
44         informatica_acessorios  1549372.59
54               moveis_decoracao  1394466.93
66             relogios_presentes  1387362.45
..                            ...         ...
60                       pc_gamer     1925.01
15                casa_conforto_2     1710.54
17              cds_dvds_musicais     1199.43
37  fashion_roupa_infanto_juvenil      718.98
67             seguros_e_servicos      324.51

[73 rows x 2 columns]


cama_mesa_banho với doanh thu là  1692714.28 là danh mục có doanh thu cao nhất trùng với danh mục có nhiều đơn nhất

 relogios_presentes đứng thứ 6 về lượng bán nhưng đứng thứ 5 về doanh thu cho thấy giá trị đơn hàng cao hơn trung bình


In [15]:
# phân tích khách hàng
df_customers = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_customers_dataset.csv')

# merge voiws orders
df_customer_order = df_d.merge(df_customers,on='customer_id',how='left')

# số đơn hàng theo bang
customer_by_state = (
    df_customer_order.groupby('customer_state')
    .agg(total_orders = ('order_id','count'))
    .reset_index()
    .sort_values('total_orders',ascending=False)

)

customer_by_state['pct']= (
    customer_by_state['total_orders']/customer_by_state['total_orders'].sum()*100
).round(2)
print(customer_by_state.head(10))

   customer_state  total_orders    pct
25             SP         40501  41.98
18             RJ         12350  12.80
10             MG         11354  11.77
22             RS          5345   5.54
17             PR          4923   5.10
23             SC          3546   3.68
4              BA          3256   3.37
6              DF          2080   2.16
7              ES          1995   2.07
8              GO          1957   2.03


sp chiếm 40.501 đơn - 41.98% tổng đơn hàng, cần tập trung mạnh ở thị trường này

bên cạnh đó còn có các thị trương tiềm năng là RJ(12.8%) và MG(11.77%)

In [16]:
# phân tích review và satisfaction
df_reviews = pd.read_csv('/kaggle/input/brazilian-ecommerce/olist_order_reviews_dataset.csv')
# phân bổ điểm review
print(df_reviews['review_score'].value_counts().sort_index())
# phần trăm số lượng để dễ so sánh
print(df_reviews['review_score'].value_counts(normalize=True).sort_index())

# điểm trung bình
print(f'điểm trung bình:{df_reviews["review_score"].mean():.2f}')

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64
review_score
1    0.115133
2    0.031756
3    0.082430
4    0.192917
5    0.577763
Name: proportion, dtype: float64
điểm trung bình:4.09


review cho 5 sao chiếm 57.7763%

điểm trung bình 4.09 khá tốt

nhưng có 11.5133% review 1 sao -> có liên quan đến giao hàng chậm không?



In [17]:
# merge reviews với orders đã có delivery_days
df_reviews_d = df_d.merge(df_reviews[['order_id','review_score']],on='order_id',how='left')

# thời gian giao hàng trung bình theo điểm review
review_delivery_days = (
    df_reviews_d.groupby('review_score')
    .agg(avg_delivery_days=('d_day','mean'))
    .reset_index()
)
print(review_delivery_days)
#

   review_score  avg_delivery_days
0           1.0          20.849973
1           2.0          16.194832
2           3.0          13.793242
3           4.0          11.848054
4           5.0          10.224154


giao hàng chậm là nguyên nhân chính khiến khách cho đánh giá thấp

**Đề xuất cải thiện logistics thay vì tập trung vào marketing**

In [18]:
# xuất các bảng phân tích ra file
monthly_orders.to_csv('monthly_orders.csv',index=False)
monthly_revenue.to_csv('monthly_revenue.csv',index=False)
top_categories.to_csv('top_categories.csv',index=False)
category_revenue.to_csv('category_revenue.csv',index=False)
customer_by_state.to_csv('customer_by_state.csv',index=False)
review_delivery_days.to_csv('review_delivery_days.csv',index=False)
print('xuất file thành công')

xuất file thành công
